In [0]:
from pyspark.sql.functions import *

## read silver tables

In [0]:
orders = spark.read.format("delta").load(
"/Volumes/ecommerce_catalog/ecommerce_schema/ecommerce_volume/silver/orders")

customers = spark.read.format("delta").load(
"/Volumes/ecommerce_catalog/ecommerce_schema/ecommerce_volume/silver/customers")

products = spark.read.format("delta").load(
"/Volumes/ecommerce_catalog/ecommerce_schema/ecommerce_volume/silver/products")

payments = spark.read.format("delta").load(
"/Volumes/ecommerce_catalog/ecommerce_schema/ecommerce_volume/silver/payments")

# star schema

## fact sales

In [0]:
fact_sales = orders \
.join(customers,"customer_id") \
.join(products,"product_id") \
.join(payments,"order_id")

In [0]:
fact_sales.display()

order_id,product_id,customer_id,quantity,price,order_date,name,city,state,product_name,category,payment_id,payment_method,amount
1003,103,3,3,200,2025-01-03,Bob,Dallas,TX,Book,Education,3,Debit Card,600
1002,102,2,1,800,2025-01-02,Alice,Chicago,IL,Phone,Electronics,2,UPI,800
1001,101,1,2,500,2025-01-01,John,New York,NY,Laptop,Electronics,1,Credit Card,1000


## revenue calculation

In [0]:
fact_sales = fact_sales.withColumn("revenue",col("quantity") * col("price"))

## save fact table

In [0]:
fact_sales.write.format("delta").mode("overwrite") \
.save("/Volumes/ecommerce_catalog/ecommerce_schema/ecommerce_volume/gold/fact_sales")

## customer dimension

In [0]:
dim_customer = customers.select("customer_id","name","city","state")

dim_customer.write.format("delta").mode("overwrite") \
.save("/Volumes/ecommerce_catalog/ecommerce_schema/ecommerce_volume/gold/dim_customer")

## product dimension

In [0]:
dim_product = products.select("product_id","product_name","category")

dim_product.write.format("delta").mode("overwrite") \
.save("/Volumes/ecommerce_catalog/ecommerce_schema/ecommerce_volume/gold/dim_product")